# Day 2 — SQL Analysis

This notebook uses SQL to analyze the cleaned Road Accidents in India dataset.

The goal is to show how the same cleaned data can be queried using SQL for common data analyst questions such as:

1. Top States/UTs by accident count
2. Top States/UTs by accident rate per lakh population
3. Ranking differences between raw counts and normalized rates
4. Biggest increases and decreases from 2023 to 2024
5. Region-wise accident patterns
6. Year-wise accident trends

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path
from IPython.display import display


print("Libraries imported successfully")

Libraries imported successfully


In [2]:
processed_dir = Path("../data/processed")

accidents_path = processed_dir / "accidents_cleaned.csv"
trend_path = processed_dir / "accidents_trend_long.csv"
regional_path = processed_dir / "regional_summary_2024.csv"

accidents = pd.read_csv(accidents_path)
trend_data = pd.read_csv(trend_path)
regional_summary = pd.read_csv(regional_path)

print("accidents:", accidents.shape)
print("trend_data:", trend_data.shape)
print("regional_summary:", regional_summary.shape)

display(accidents.head())

accidents: (36, 21)
trend_data: (144, 5)
regional_summary: (7, 8)


,serial_no,state_ut,accidents_2021,accidents_2022,accidents_2023,accidents_2024,accidents_2024_rank,share_2021_pct,share_2022_pct,share_2023_pct,...,accidents_per_lakh_pop_2021,accidents_per_lakh_pop_2022,accidents_per_lakh_pop_2023,accidents_per_lakh_pop_2024,accidents_per_10k_vehicles_2022,accidents_per_10k_km_roads_2022,vehicle_rate_available,yoy_change_2024_pct,cagr_2021_2024_pct,region
0,1,Andhra Pradesh,21556,21249,19949,19557,9,5.2,4.6,4.2,...,40.8,40.1,37.5,36.7,14.2,1098.5,True,-1.97,-3.19,South
1,2,Arunachal Pradesh,283,227,287,277,27,0.1,0.0,0.1,...,18.5,14.7,18.4,17.6,6.2,42.8,True,-3.48,-0.71,Northeast
2,3,Assam,7411,7023,7421,7848,16,1.8,1.5,1.5,...,21.1,19.9,20.8,21.8,15.6,140.1,True,5.75,1.93,Northeast
3,4,Bihar,9553,10801,11014,11610,14,2.3,2.3,2.3,...,7.8,8.6,8.7,9.0,9.2,357.5,True,5.41,6.72,East
4,5,Chhattisgarh,12375,13279,13468,14857,11,3.0,2.9,2.8,...,42.0,44.5,44.6,48.7,17.0,1171.9,True,10.31,6.28,Central


In [3]:
conn = sqlite3.connect(":memory:")

accidents.to_sql(
    "accidents",
    conn,
    index=False,
    if_exists="replace"
)

trend_data.to_sql(
    "trend_data",
    conn,
    index=False,
    if_exists="replace"
)

regional_summary.to_sql(
    "regional_summary",
    conn,
    index=False,
    if_exists="replace"
)

print("SQLite database created successfully")

SQLite database created successfully


In [4]:
def run_query(query):
    return pd.read_sql_query(query, conn)

In [5]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

run_query(query)

,name
0,accidents
1,trend_data
2,regional_summary


In [6]:
query = """
SELECT *
FROM accidents
LIMIT 5;
"""

run_query(query)

,serial_no,state_ut,accidents_2021,accidents_2022,accidents_2023,accidents_2024,accidents_2024_rank,share_2021_pct,share_2022_pct,share_2023_pct,...,accidents_per_lakh_pop_2021,accidents_per_lakh_pop_2022,accidents_per_lakh_pop_2023,accidents_per_lakh_pop_2024,accidents_per_10k_vehicles_2022,accidents_per_10k_km_roads_2022,vehicle_rate_available,yoy_change_2024_pct,cagr_2021_2024_pct,region
0,1,Andhra Pradesh,21556,21249,19949,19557,9,5.2,4.6,4.2,...,40.8,40.1,37.5,36.7,14.2,1098.5,1,-1.97,-3.19,South
1,2,Arunachal Pradesh,283,227,287,277,27,0.1,0.0,0.1,...,18.5,14.7,18.4,17.6,6.2,42.8,1,-3.48,-0.71,Northeast
2,3,Assam,7411,7023,7421,7848,16,1.8,1.5,1.5,...,21.1,19.9,20.8,21.8,15.6,140.1,1,5.75,1.93,Northeast
3,4,Bihar,9553,10801,11014,11610,14,2.3,2.3,2.3,...,7.8,8.6,8.7,9.0,9.2,357.5,1,5.41,6.72,East
4,5,Chhattisgarh,12375,13279,13468,14857,11,3.0,2.9,2.8,...,42.0,44.5,44.6,48.7,17.0,1171.9,1,10.31,6.28,Central


## 1. Top 10 States/UTs by 2024 Accident Count

This query identifies the States/UTs with the highest number of reported road accidents in 2024.

This is a raw-volume ranking, so larger States may appear higher because of population size, vehicle density and road activity.

In [7]:
query = """
SELECT
    state_ut,
    region,
    accidents_2024,
    accidents_2024_rank,
    share_2024_pct,
    accidents_per_lakh_pop_2024
FROM accidents
ORDER BY accidents_2024 DESC
LIMIT 10;
"""

top_10_sql_raw = run_query(query)

display(top_10_sql_raw)

,state_ut,region,accidents_2024,accidents_2024_rank,share_2024_pct,accidents_per_lakh_pop_2024
0,Tamil Nadu,South,67526,1,13.8,87.6
1,Madhya Pradesh,Central,56669,2,11.6,64.7
2,Kerala,South,48834,3,10.0,136.0
3,Uttar Pradesh,North,46052,4,9.4,19.3
4,Karnataka,South,43062,5,8.8,63.2
5,Maharashtra,West,36118,6,7.4,28.4
6,Telangana,South,25986,7,5.3,67.9
7,Rajasthan,North,24838,8,5.1,30.3
8,Andhra Pradesh,South,19557,9,4.0,36.7
9,Gujarat,West,15588,10,3.2,21.5


### SQL Insight 1

The SQL query shows that Tamil Nadu recorded the highest number of road accidents in 2024, followed by Madhya Pradesh, Kerala, Uttar Pradesh and Karnataka.

This confirms the raw accident-count ranking previously found using Python. However, this is only a volume-based view. To understand road-safety risk better, the next query will compare States/UTs using accidents per lakh population.

## 2. Top 10 States/UTs by Accidents per Lakh Population

This query ranks States/UTs by population-normalized accident rate in 2024.

Unlike raw accident count, this metric helps compare accident burden relative to population size.

In [8]:
query = """
SELECT
    state_ut,
    region,
    accidents_2024,
    accidents_2024_rank,
    accidents_per_lakh_pop_2024
FROM accidents
ORDER BY accidents_per_lakh_pop_2024 DESC
LIMIT 10;
"""

top_10_sql_rate = run_query(query)

display(top_10_sql_rate)

,state_ut,region,accidents_2024,accidents_2024_rank,accidents_per_lakh_pop_2024
0,Goa,West,2682,21,169.4
1,Kerala,South,48834,3,136.0
2,Tamil Nadu,South,67526,1,87.6
3,Ladakh,Union Territory,264,29,87.4
4,Puducherry,Union Territory,1431,24,85.0
5,Telangana,South,25986,7,67.9
6,Madhya Pradesh,Central,56669,2,64.7
7,Karnataka,South,43062,5,63.2
8,Chhattisgarh,Central,14857,11,48.7
9,Jammu and Kashmir,Union Territory,5808,18,42.4


### SQL Insight 2

The population-normalized ranking gives a different view from the raw accident-count ranking.

While Tamil Nadu leads in total accidents, Goa ranks highest by accidents per lakh population. Kerala and Tamil Nadu also remain high in the normalized ranking, but smaller States/UTs such as Ladakh and Puducherry become more visible.

This shows why normalized metrics are important. Raw counts show accident volume, while accidents per lakh population provide a better signal of relative road-safety risk.

## 3. States/UTs That Look Riskier After Population Normalization

This query compares raw accident-count rank with population-normalized accident-rate rank.

A high positive rank difference means the State/UT ranks much higher after normalization than it does by raw accident count.

In [9]:
query = """
WITH ranked_data AS (
    SELECT
        state_ut,
        region,
        accidents_2024,
        accidents_per_lakh_pop_2024,
        RANK() OVER (
            ORDER BY accidents_2024 DESC
        ) AS raw_count_rank,
        RANK() OVER (
            ORDER BY accidents_per_lakh_pop_2024 DESC
        ) AS population_rate_rank
    FROM accidents
)

SELECT
    state_ut,
    region,
    accidents_2024,
    accidents_per_lakh_pop_2024,
    raw_count_rank,
    population_rate_rank,
    raw_count_rank - population_rate_rank AS rank_difference
FROM ranked_data
ORDER BY rank_difference DESC
LIMIT 10;
"""

riskier_after_normalization_sql = run_query(query)

display(riskier_after_normalization_sql)

,state_ut,region,accidents_2024,accidents_per_lakh_pop_2024,raw_count_rank,population_rate_rank,rank_difference
0,Ladakh,Union Territory,264,87.4,29,4,25
1,Andaman and Nicobar Islands,Union Territory,135,33.4,33,12,21
2,Goa,West,2682,169.4,21,1,20
3,Puducherry,Union Territory,1431,85.0,24,5,19
4,Sikkim,Northeast,149,21.4,32,21,11
5,Jammu and Kashmir,Union Territory,5808,42.4,18,10,8
6,Himachal Pradesh,North,2156,28.7,22,15,7
7,Mizoram,Northeast,118,9.4,35,31,4
8,Arunachal Pradesh,Northeast,277,17.6,27,24,3
9,Chhattisgarh,Central,14857,48.7,11,9,2


### SQL Insight 3

This query identifies States/UTs that appear more important after adjusting for population.

Smaller States/UTs such as Ladakh, Andaman and Nicobar Islands, Goa, Puducherry and Sikkim move much higher in the normalized ranking than in the raw accident-count ranking.

This reinforces the idea that low total accident count does not always mean low road-safety risk.

## 4. States/UTs That Look Less Extreme After Population Normalization

This query identifies States/UTs that rank high by raw accident count but lower after adjusting for population.

A large negative rank difference suggests that the State/UT's high accident count may be partly driven by scale.

In [10]:
query = """
WITH ranked_data AS (
    SELECT
        state_ut,
        region,
        accidents_2024,
        accidents_per_lakh_pop_2024,
        RANK() OVER (
            ORDER BY accidents_2024 DESC
        ) AS raw_count_rank,
        RANK() OVER (
            ORDER BY accidents_per_lakh_pop_2024 DESC
        ) AS population_rate_rank
    FROM accidents
)

SELECT
    state_ut,
    region,
    accidents_2024,
    accidents_per_lakh_pop_2024,
    raw_count_rank,
    population_rate_rank,
    raw_count_rank - population_rate_rank AS rank_difference
FROM ranked_data
ORDER BY rank_difference ASC
LIMIT 10;
"""

less_extreme_after_normalization_sql = run_query(query)

display(less_extreme_after_normalization_sql)

,state_ut,region,accidents_2024,accidents_per_lakh_pop_2024,raw_count_rank,population_rate_rank,rank_difference
0,Uttar Pradesh,North,46052,19.3,4,23,-19
1,Bihar,East,11610,9.0,14,33,-19
2,West Bengal,East,13702,13.8,12,26,-14
3,Maharashtra,West,36118,28.4,6,16,-10
4,Gujarat,West,15588,21.5,10,20,-10
5,Jharkhand,East,5196,13.0,20,29,-9
6,Rajasthan,North,24838,30.3,8,14,-6
7,Manipur,Northeast,299,9.2,26,32,-6
8,Meghalaya,Northeast,269,8.0,28,34,-6
9,Madhya Pradesh,Central,56669,64.7,2,7,-5


### SQL Insight 4

This query shows States/UTs that look less extreme after population normalization.

Some large States may rank high by total accidents but move lower when accidents are compared relative to population size.

This does not mean these States are safe. It means raw volume and normalized risk are answering different analytical questions.

## 5. Top 10 States/UTs by Absolute Increase from 2023 to 2024

This query identifies where the number of road accidents increased the most in absolute terms.

Absolute change helps show which States/UTs added the largest number of accidents to the national total.

In [11]:
query = """
SELECT
    state_ut,
    region,
    accidents_2023,
    accidents_2024,
    accidents_2024 - accidents_2023 AS absolute_change_2024,
    yoy_change_2024_pct
FROM accidents
ORDER BY absolute_change_2024 DESC
LIMIT 10;
"""

top_10_sql_absolute_increase = run_query(query)

display(top_10_sql_absolute_increase)

,state_ut,region,accidents_2023,accidents_2024,absolute_change_2024,yoy_change_2024_pct
0,Telangana,South,22903,25986,3083,13.46
1,Uttar Pradesh,North,44534,46052,1518,3.41
2,Chhattisgarh,Central,13468,14857,1389,10.31
3,Madhya Pradesh,Central,55327,56669,1342,2.43
4,Maharashtra,West,35243,36118,875,2.48
5,Kerala,South,48091,48834,743,1.54
6,Bihar,East,11014,11610,596,5.41
7,Assam,Northeast,7421,7848,427,5.75
8,Odisha,East,11992,12375,383,3.19
9,Tamil Nadu,South,67213,67526,313,0.47


### SQL Insight 5

The absolute-increase ranking highlights where accident volume increased the most from 2023 to 2024.

This is useful for understanding which States/UTs contributed most to the increase in national accident burden.

## 6. Top 10 States/UTs by Absolute Decrease from 2023 to 2024

This query identifies where the number of road accidents decreased the most in absolute terms.

Absolute decrease helps identify States/UTs that reduced accident volume most strongly.

In [12]:
query = """
SELECT
    state_ut,
    region,
    accidents_2023,
    accidents_2024,
    accidents_2024 - accidents_2023 AS absolute_change_2024,
    yoy_change_2024_pct
FROM accidents
ORDER BY absolute_change_2024 ASC
LIMIT 10;
"""

top_10_sql_absolute_decrease = run_query(query)

display(top_10_sql_absolute_decrease)

,state_ut,region,accidents_2023,accidents_2024,absolute_change_2024,yoy_change_2024_pct
0,Gujarat,West,16349,15588,-761,-4.65
1,Haryana,North,10463,9806,-657,-6.28
2,Jammu and Kashmir,Union Territory,6298,5808,-490,-7.78
3,Andhra Pradesh,South,19949,19557,-392,-1.97
4,Karnataka,South,43440,43062,-378,-0.87
5,Punjab,North,6269,6063,-206,-3.29
6,Delhi,Union Territory,5834,5657,-177,-3.03
7,Nagaland,Northeast,303,129,-174,-57.43
8,Goa,West,2846,2682,-164,-5.76
9,Jharkhand,East,5315,5196,-119,-2.24


### SQL Insight 6

The absolute-decrease ranking shows where accident counts fell the most from 2023 to 2024.

This view is useful because even a small percentage change in a large State can represent a large number of accidents.

## 7. Top 10 States/UTs by Percentage Increase from 2023 to 2024

This query ranks States/UTs by year-over-year percentage increase.

Percentage change helps detect sharp relative changes, especially in smaller States/UTs where absolute accident counts may be low.

In [13]:
query = """
SELECT
    state_ut,
    region,
    accidents_2023,
    accidents_2024,
    accidents_2024 - accidents_2023 AS absolute_change_2024,
    yoy_change_2024_pct
FROM accidents
ORDER BY yoy_change_2024_pct DESC
LIMIT 10;
"""

top_10_sql_pct_increase = run_query(query)

display(top_10_sql_pct_increase)

,state_ut,region,accidents_2023,accidents_2024,absolute_change_2024,yoy_change_2024_pct
0,Meghalaya,Northeast,223,269,46,20.63
1,Telangana,South,22903,25986,3083,13.46
2,Mizoram,Northeast,106,118,12,11.32
3,Chhattisgarh,Central,13468,14857,1389,10.31
4,Puducherry,Union Territory,1308,1431,123,9.40
5,Assam,Northeast,7421,7848,427,5.75
6,Bihar,East,11014,11610,596,5.41
7,Uttar Pradesh,North,44534,46052,1518,3.41
8,Uttarakhand,North,1691,1747,56,3.31
9,Odisha,East,11992,12375,383,3.19


### SQL Insight 7

The percentage-increase ranking gives a different picture from the absolute-increase ranking.

Smaller States/UTs can appear higher because even a modest numerical increase may represent a large percentage change compared with their previous-year base.

This is why both absolute and percentage change should be reviewed together.

## 8. Top 10 States/UTs by Percentage Decrease from 2023 to 2024

This query ranks States/UTs by year-over-year percentage decrease.

Percentage decrease helps identify places where accident counts fell sharply relative to their own 2023 levels.

In [14]:
query = """
SELECT
    state_ut,
    region,
    accidents_2023,
    accidents_2024,
    accidents_2024 - accidents_2023 AS absolute_change_2024,
    yoy_change_2024_pct
FROM accidents
ORDER BY yoy_change_2024_pct ASC
LIMIT 10;
"""

top_10_sql_pct_decrease = run_query(query)

display(top_10_sql_pct_decrease)

,state_ut,region,accidents_2023,accidents_2024,absolute_change_2024,yoy_change_2024_pct
0,Lakshadweep,Union Territory,1,0,-1,-100.00
1,Nagaland,Northeast,303,129,-174,-57.43
2,Manipur,Northeast,398,299,-99,-24.87
3,Sikkim,Northeast,182,149,-33,-18.13
4,Dadra and Nagar Haveli and Daman and Diu,Union Territory,182,152,-30,-16.48
5,Ladakh,Union Territory,289,264,-25,-8.65
6,Jammu and Kashmir,Union Territory,6298,5808,-490,-7.78
7,Chandigarh,Union Territory,182,169,-13,-7.14
8,Haryana,North,10463,9806,-657,-6.28
9,Goa,West,2846,2682,-164,-5.76


### SQL Insight 8

The percentage-decrease ranking highlights States/UTs where accident counts fell most sharply relative to their 2023 levels.

This is useful for spotting improvement, but the results should be interpreted carefully for smaller States/UTs because small absolute changes can create large percentage movements.

## 9. Region-Wise Accident Summary for 2024

This query summarizes accident patterns by region.

It compares total accident volume, average accidents per lakh population and year-over-year change across regions.

In [15]:
query = """
SELECT
    region,
    state_ut_count,
    total_accidents_2024,
    avg_accidents_per_lakh_pop_2024,
    median_accidents_per_lakh_pop_2024,
    total_absolute_change_2024,
    avg_yoy_change_2024_pct
FROM regional_summary
ORDER BY total_accidents_2024 DESC;
"""

regional_summary_sql = run_query(query)

display(regional_summary_sql)

,region,state_ut_count,total_accidents_2024,avg_accidents_per_lakh_pop_2024,median_accidents_per_lakh_pop_2024,total_absolute_change_2024,avg_yoy_change_2024_pct
0,South,5,204965,78.28,67.90,3369,2.53
1,North,6,90662,24.15,24.15,758,-1.10
2,Central,2,71526,56.70,56.70,2731,6.37
3,West,3,54388,73.10,28.40,-50,-2.64
4,East,4,42883,15.60,13.40,767,1.42
5,Union Territory,8,13616,37.36,29.70,-621,-17.41
6,Northeast,8,9667,13.36,11.60,170,-8.26


### SQL Insight 9

The South region recorded the highest total number of accidents in 2024. It also showed the highest average accidents per lakh population, making it important from both a volume and normalized-risk perspective.

The regional view confirms that accident patterns differ depending on whether we compare total accidents, average normalized rate or year-over-year change.

## 10. National Accident Trend from 2021 to 2024

This query uses the long-format trend table to calculate total accidents by year.

It helps verify the overall national accident trend across the four-year period.

In [16]:
query = """
SELECT
    year,
    SUM(accident_count) AS national_accident_total
FROM trend_data
GROUP BY year
ORDER BY year;
"""

national_trend_sql = run_query(query)

display(national_trend_sql)

,year,national_accident_total
0,2021,412432
1,2022,461312
2,2023,480583
3,2024,487707


### SQL Insight 10

The national trend shows that total reported road accidents increased from 2021 to 2024.

This SQL result also confirms that the long-format trend table can be used for year-wise analysis and dashboard visuals.

## SQL Analysis Summary

The SQL analysis confirmed the key findings from the Python EDA notebook.

Key takeaways:

1. Tamil Nadu recorded the highest number of road accidents in 2024 by raw count.
2. Goa ranked highest by accidents per lakh population, showing that normalized metrics tell a different story from raw counts.
3. Smaller States/UTs such as Ladakh, Andaman and Nicobar Islands, Puducherry and Sikkim became more visible after population normalization.
4. Absolute change and percentage change highlighted different patterns from 2023 to 2024.
5. The South region had the highest total accident count and also the highest average accidents per lakh population.
6. National accident totals increased from 2021 to 2024.

Overall, the SQL queries showed that the cleaned dataset can be analyzed using standard data analyst workflows such as ranking, filtering, aggregation, grouping, and trend analysis.